# Data Retrieval
---

To fit a TESS light curve with Allesfitter, start with a product from MAST and prepare the three columns the fit needs: time, flux, and flux uncertainty. We use Lightkurve to find and download a target's light curve, then trim and normalize it for Allesfitter.

We'll use HD 189733 (TOI 4470.01; TIC 256364928) as the example. Swap in another TESS TIC ID to follow the same steps for a different target. For instruments besides Kepler or TESS, use a mission-appropriate retrieval method.

In [ ]:
import lightkurve as lk
import pandas as pd

#### Retrieving the Data

Lightkurve searches MAST for TESS light curves by TIC ID. Replace `TIC#` in the code cell with your target's TIC ID, then run it to see the available products.



In [ ]:
search_result = lk.search_lightcurve("TIC#")
print(search_result)

Inspect the results for the target, mission, sector, and cadence you want. The next cell downloads the first result with `search_result[0]` and saves it to `allesfit/inst.csv`. Create the `allesfit/` directory first, or change the output path in that cell. If the first result is not your intended product, change its index before downloading.



In [ ]:
lc = search_result[0].download()
lc.to_csv('allesfit/inst.csv')

Take a look at the csv file that you created. Currently, it has more columns of data than are needed. To produce fits for light curves or phase curves, only time, flux, and flux_err is needed.

In [ ]:
file = pd.read_csv('allesfit/inst.csv')
data = file.drop(columns=['timecorr','cadenceno','centroid_col','centroid_row','sap_flux','sap_flux_err','sap_bkg','sap_bkg_err','pdcsap_flux','pdcsap_flux_err','quality','psf_centr1','psf_centr1_err','psf_centr2','psf_centr2_err','mom_centr1','mom_centr1_err','mom_centr2','mom_centr2_err','pos_corr1','pos_corr2'])
data.rename(columns = {'time':'#time'}, inplace = True)
data.to_csv('allesfit/inst.csv', index=False)

#### Formatting the Data

Now we have the data that we need, but before we run Allesfitter, a couple of adjustments have to be made.

In [ ]:
data = pd.read_csv('allesfit/inst.csv')
data

Drop rows with missing values in `#time`, `flux`, or `flux_err` before fitting. Try writing the pandas operation in the empty cell below, then compare it with the example solution.


In [ ]:
### Write your code here to delete rows with NaN values. Check the box below for a hint/solution



In [ ]:
#There are many different ways that you could have completed this. Here are two of them:

#version 1:
for index, row in data.iterrows():

    for column in row:

        if column=='':
            data = data.drop(row)


#version 2:
data.dropna(subset=['#time','flux','flux_err'], inplace=True)

#print the data to see if/how it has changed
data

In [ ]:
data.to_csv('allesfit/inst.csv', index=False)

Divide `flux` and its reported uncertainty, `flux_err`, by the median flux. The resulting light curve is in relative-flux units, and each pointwise uncertainty is scaled by the same factor.


In [ ]:
data = pd.read_csv('allesfit/inst.csv')

reference_flux = data['flux'].median()

data['flux'] = data['flux'] / reference_flux

##propogating error 
data['flux_err'] = data['flux_err'] / reference_flux

data.to_csv('allesfit/inst.csv',index=False)

Once missing measurements are removed and flux and ``flux_err`` are divided by the same median, ``inst.csv`` has the core columns for an Allesfitter fit. Try another TESS target by changing the TIC ID above, then continue to the setup tutorial to configure your fit.

This tutorial uses Lightkurve (Lightkurve Collaboration, 2018) and pandas (The Pandas Development Team, 2010).